In [1]:
from rdkit import Chem
from rdkit.Chem import FilterCatalog, Descriptors, Crippen, rdMolDescriptors, BRICS
import json
from tqdm import tqdm
import random
from collections import namedtuple
import numpy as np
from pathlib import Path
import pickle
import sys

from sltools import property_tools
from sltools.property_tools import OUT_OF_RANGE, UNDESIRABLE_PATTERNS, CORRECT_PYRROLE, COVALENT_WARHEADS, params, PAINS_catalog
from sltools.inference_tools import InferenceObject

In [2]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [3]:
temperature = 1.0
model_path = "THGLab/Llama-3.1-8B-SmileyLlama-1.1"
#model_path = "THGLab/Llama-3.1-8B-SmileyLlama-1.1-Prompt-Following"
system_text = "You love and excel at generating SMILES strings of drug-like molecules"
tokenizer_path = model_path
num_return_sequences = 16
max_new_tokens = 128
io = InferenceObject(model_path, tokenizer_path, num_return_sequences, temperature, max_new_tokens)

PyTorch Using cuda device with 4 GPUs


2025-06-20 14:20:33.335588: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-20 14:20:34.584760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-20 14:20:34.645661: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-20 14:20:34.650342: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-20 14:20:34.721196: I tensorflow/core/platform/cpu_feature_guar

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [15]:


from property_tools import has_pains_alert, has_bad_ring, check_valid_pattern

def check_for_covalent_warheads(input):
    """
    return True if it contains any covalent warheads
    """
    if type(input) is str:
        mol = Chem.MolFromSmiles(input)
    else:
        mol = input
    for smarts in COVALENT_WARHEADS:
        if mol.HasSubstructMatch(Chem.MolFromSmarts(smarts)):
            return True
    return False

def get_covalent_warheads(input):
    """
    return True if it contains any covalent warheads
    """
    warhead_list = []
    if type(input) is str:
        mol = Chem.MolFromSmiles(input)
    else:
        mol = input
    mol = Chem.AddHs(mol)
    for warhead_name, smarts in COVALENT_WARHEADS.items():
        if mol.HasSubstructMatch(Chem.MolFromSmarts(smarts)):
            warhead_list.append(warhead_name)
    return warhead_list


def check_for_macrocycle(mol):
    """
    return True if it contains any macrocycles
    """
    ring_info = mol.GetRingInfo()
    for ring in ring_info.AtomRings():
        if len(ring) >= 8:
            return True
    return False

def is_substructure(smiles_a, smiles_b):
    # Convert SMILES strings to RDKit molecules
    mol_a = Chem.MolFromSmiles(smiles_a)
    mol_b = Chem.MolFromSmiles(smiles_b)

    if mol_a is None or mol_b is None:
        return False  # Invalid SMILES strings

    # Check if A contains a ghost atom
    if '*' in smiles_a:
        print("Ghost atom detected!")
        canonical_smiles_a = Chem.MolToSmiles(mol_a, canonical=True)
        frags = BRICS.BRICSDecompose(mol_b, returnMols=True, singlePass=True)
        for mol in frags:
            rw_mol = Chem.RWMol(mol)
            # Iterate over atoms in the molecule
            for atom in rw_mol.GetAtoms():
                # Check if the atom is a special marker (atomic number 0 and a specific isotope)
                if atom.GetAtomicNum() == 0:
                    # Replace the special marker with a hydrogen atom
                    new_atom = Chem.Atom(0)
                    rw_mol.ReplaceAtom(atom.GetIdx(), new_atom)
            canonical_smiles_frag = Chem.MolToSmiles(rw_mol, canonical=True)
            if canonical_smiles_frag == canonical_smiles_a:
                return True
        return False
    else:
        # Standard substructure check
        return mol_b.HasSubstructMatch(mol_a)

def check_molecular_properties(smiles, property_constraints, substructure=False):
    # Create RDKit molecule object
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False  # Invalid SMILES
    # Define property calculation functions
    property_functions = {
        'hbd': Descriptors.NumHDonors,
        'hba': Descriptors.NumHAcceptors,
        'mw': Descriptors.ExactMolWt,
        'logp': Crippen.MolLogP,
        'rotb': Descriptors.NumRotatableBonds,
        'fracsp3': Descriptors.FractionCSP3,
        'tpsa': Descriptors.TPSA,
        'macrocycle': check_for_macrocycle,
        'no_undesirable_smarts': check_valid_pattern,
        'cov_warhead': get_covalent_warheads,
        'formula': rdMolDescriptors.CalcMolFormula,
    }
    # Check each constraint
    for prop, constraint in property_constraints.items():
        if prop not in property_functions:
            raise ValueError(f"Unsupported property: {prop}")
        value = property_functions[prop](mol)
        print(prop, value)
        if isinstance(constraint, tuple):
            operator, limit = constraint
            if operator == '<=':
                if not value <= limit:
                    return False
            elif operator == '<':
                if not value < limit:
                    return False
            elif operator == '>=':
                if not value >= limit:
                    return False
            elif operator == '>':
                if not value > limit:
                    return False
            elif operator == '==':
                if not value == limit:
                    return False
            elif operator == 'between':
                if not limit[0] <= value <= limit[1]:
                    return False
            elif operator == 'is a superset of':
                assert isinstance(limit, set) and isinstance(value, list), "need to pass in a set for limit and a list for value"
                set_value = set(value)
                if not set_value >= limit:
                    return False
            else:
                raise ValueError(f"Unsupported operator: {operator}")
        else:
            raise ValueError(f"Invalid constraint format for {prop}")
    if substructure != False:
        return is_substructure(substructure, smiles)
    return True

with open('essential_frags.csv', 'r') as frag_csv:
    lines = frag_csv.readlines()
substructs = [l.split(',')[0] for l in lines[1:]]

properties = ['exactly ' + str(k) + ' H-bond donors' for k in range(0, 6)]
properties += ['exactly ' + str(k) + ' H-bond acceptors' for k in range(0, 11)]
properties += ['<= ' + str(k) + ' H-bond donors' for k in [3,4,5,7]]
properties += ['<= ' + str(k) + ' H-bond acceptors' for k in [3,4,5,10,15]]
properties += ['<= ' + str(k) + ' molecular weight' for k in [300,400,500,600]]
properties += ['<= ' + str(k) + ' logP' for k in [3,4,5,6]]
properties += ['<= ' + str(k) + ' Rotatable bonds' for k in [7,10]]
properties += ['> 10 Rotatable bonds']
properties += ['> ' + str(k) + ' Fraction sp3' for k in [0.4, 0.5, 0.6]]
properties += ['< 0.4 Fraction sp3']
properties += ['<= ' + str(k) + ' TPSA' for k in [90, 140, 200]]
properties += ['a macrocycle', 'no macrocycles']
properties += ['lacks bad SMARTS', 'has bad SMARTS']
properties += ['lacks covalent warheads'] + ['has covalent warheads ('+warhead_name+')' for warhead_name in COVALENT_WARHEADS.keys()]
properties += ['a substructure of ' + s for s in substructs]
properties += ['<= 5 H-bond donors, <= 10 H-bond acceptors, <= 500 molecule, <= 5 logP']
properties += ['<= 3 H-bond donors, <= 3 H-bond acceptors, <= 300 molecule, <= 3 logP']

constraint_list = [{'hbd': ('==', k)} for k in range(0, 6)]
constraint_list += [{'hba': ('==', k)} for k in range(0, 11)]
constraint_list += [{'hbd': ('<=', k)} for k in [3, 4, 5, 7]]
constraint_list += [{'hba': ('<=', k)} for k in [3, 4, 5, 10, 15]]
constraint_list += [{'mw': ('<=', k)} for k in [300, 400, 500, 600]]
constraint_list += [{'logp': ('<=', k)} for k in [3, 4, 5, 6]]
constraint_list += [{'rotb': ('<=', k)} for k in [7, 10]]
constraint_list += [{'rotb': ('>', 10)}]
constraint_list += [{'fracsp3': ('>', k)} for k in [0.4, 0.5, 0.6]]
constraint_list += [{'fracsp3': ('<', 0.4)}]
constraint_list += [{'tpsa': ('<=', k)} for k in [90, 140, 200]]
constraint_list += [{'macrocycle': ('==', True)}, {'macrocycle': ('==', False)}]
constraint_list += [{'no_undesirable_smarts': ('==', True)}, {'no_undesirable_smarts': ('==', False)}]
constraint_list += [{'cov_warhead': ('==', [])}] + [{'cov_warhead': ('is a superset of', {warhead_name})} for warhead_name in COVALENT_WARHEADS.keys()]
constraint_list += [{'substruct': s} for s in substructs]

constraint_list += [{'mw': ('<=', 500), 'hbd': ('<=', 5), 'hba': ('<=', 10), 'logp': ('<=', 5)}]*5
constraint_list += [{'mw': ('<=', 300), 'hbd': ('<=', 3), 'hba': ('<=', 3), 'logp': ('<=', 3)}]*5


In [16]:
user_text = 'Output a SMILES string for a drug like molecule with the following properties: a substructure of O=C(O)c1ccc(C(F)(F)F)cc1, <= 500 Molecular weight, <=5 LogP, <= 5 H-bond donors, <= 10 H-bond acceptors'
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]*4
raw_results = []
strings = io.generate_strings(prompts, generation_params={"temperature":temperature}, disable_tqdm=True)
for s in strings[0:1000]:
    raw_results += s[1]
prompt_results = []
for raw_result in raw_results:
    valid = check_molecular_properties(raw_result, {'mw': ('<=', 500), 'logp': ('<=', 5), 'hbd': ('<=', 5), 'hba': ('<=', 10)}, substructure='O=C(O)c1ccc(C(F)(F)F)cc1')
    prompt_results.append((raw_result, valid))

mw 430.114041684
logp 5.577200000000002
mw 480.150821116
logp 2.8640000000000017
hbd 0
hba 7
mw 415.103142652
logp 4.376680000000004
hbd 0
hba 5
mw 395.09805727199995
logp 3.437800000000002
hbd 2
hba 4
mw 488.07661061999994
logp 4.982820000000002
hbd 1
hba 8
mw 445.0919263359999
logp 2.889100000000001
hbd 2
hba 7
mw 470.1123274319999
logp 3.675500000000003
hbd 1
hba 5
mw 317.123878096
logp 2.3189000000000006
hbd 1
hba 4
mw 190.02416406
logp 2.4036
hbd 1
hba 1
mw 531.1981056559999
mw 434.09503755600014
logp 2.03892
hbd 3
hba 7
mw 393.043577088
logp 4.8642
hbd 3
hba 3
mw 504.150821116
mw 234.0139933
logp 2.1018
hbd 2
hba 2
mw 402.11912706400005
logp 3.8079000000000027
hbd 1
hba 4
mw 459.14059077999997
logp 5.013500000000004
mw 411.08307590399994
logp 4.5895800000000015
hbd 2
hba 4
mw 495.1868722759999
logp 3.9394000000000036
hbd 1
hba 7
mw 481.186478352
logp 6.009500000000006
mw 190.02416406
logp 2.4036
hbd 1
hba 1
mw 397.16377083599997
logp 1.6990999999999996
hbd 3
hba 5
mw 392.07840512

In [18]:
print(sum([1 for p in prompt_results if p[1]])/len(prompt_results))
prompt_results

0.515625


[('FC(c1ccc(cc1)C(=O)Oc1ccc(cc1)NC(Nc1ccc(OC)cc1)=O)(F)F', False),
 ('c1cc(C(F)(F)F)ccc1C(=O)Oc1ccc(cc1C(OCC(N1CCN(CC1)C)=O)=O)OC', True),
 ('C1(C(=O)c2ccccc2)(C#N)C(CCC(C1)OC(=O)c1ccc(C(F)(F)F)cc1)=O', True),
 ('c1(ccc(cc1)C(OCC(=O)N[C@@H](C)c1ccc(C(O)=O)cc1)=O)C(F)(F)F', True),
 ('c1(-c2cccc(-c3nc(no3)C)c2)nc(sc1)NC(COC(c1ccc(cc1)C(F)(F)F)=O)=O', True),
 ('C1CN(CCN1c1nc(c(s1)COc1ccc(C(=O)O)c(c1)C(F)(F)F)C(O)=O)C', False),
 ('C(F)(F)(c1ccc(C(OCC(Nc2ccc(S(N3CCCCC3)(=O)=O)cc2)=O)=O)cc1)F', True),
 ('c1c(ccc(c1)C(OCC(CN1CCCC1)O)=O)C(F)(F)F', True),
 ('c1(ccc(C(O)=O)cc1)C(F)(F)F', True),
 ('C1CCC(CNC(c2n(c3c(cccc3)c2C(=O)O)CC)=O)CN1Cc1cc(ccc1C(F)(F)F)C(=O)O',
  False),
 ('N(C(=O)c1cc(ccc1C(F)(F)F)C(=O)O)c1nc(nc(c1)C)-n1cnc(C(N)=O)c1', True),
 ('c1cc(C(=O)O)c(cc1C(F)(F)F)Nc1cc(c(C(=O)O)cc1)C(F)(F)F', True),
 ('O=C(Nc1cc(ccc1OC)NC(c1cc(OC)c(c(OC)c1)OC)=O)c1ccc(C(F)(F)F)cc1', False),
 ('O=C(O)c1ccc(C(F)(F)F)cc1C(O)=O', True),
 ('c1cc(ccc1C(=O)OC1C2N(CCC2)C(c2c(cccc2)C=1N)=O)C(F)(F)F', True),